<a href="https://colab.research.google.com/github/Mike-Umali/NYSI-Capstone-Group-INSMS-/blob/Vector-search/Vector_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Replace the first directory with the actual path of the folder
# Second directory is to rename the lengthy directory to shorter one
# Make sure to avoid using spaces for the folder name in the google drive
!ln -s "/content/drive/MyDrive/ColabNotebooks/CapstoneVector" "/content/capstone_vector"


In [ ]:
import os

# Change the current working directory to your specific folder's shortcut
os.chdir("/content/capstone_vector")

# List files in the directory to verify
print(os.listdir())

# Read a file (example using pandas)
# import pandas as pd
# df = pd.read_csv("supplements_mock_data.csv")
# print(df.head(2))

['testingv1.ipynb', 'supplements_mock_data.csv', 'supplements_data_WITH_REAL_VECTORS.csv', 'CapstoneVector', 'supplements_data_WITH_REAL_VECTORS.gsheet', 'supplements_data_WITH_REAL_VECTORS_30.csv', 'supplements_data_WITH_REAL_VECTORS_30.gsheet', 'supplements_full_schema.csv', 'supplements_full_schema.gsheet', 'supplements_full_schema_balanced_v2.csv', 'supplements_full_schema_balanced_v2.gsheet', 'supplements_full_schema_balanced_v3.csv', 'supplements_full_schema_balanced_v3.gsheet']



# Generate Mock Data

### Add more mock data 50 rows (50/50)

`supplements_full_schema_balanced_v3.csv`

In [ ]:
import pandas as pd
import numpy as np
import json
import uuid
import random
from datetime import datetime, timedelta
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# 1. SETUP & HELPERS
# ==========================================
def get_uuid(): return str(uuid.uuid4())
def get_date(days_ago=0): return (datetime.now() - timedelta(days=days_ago)).strftime('%Y-%m-%d %H:%M:%S')

# Dose Form IDs
forms = {
    "Powder": "11111111-1111-4111-8111-111111111111",
    "Pill":   "22222222-2222-4222-8222-222222222222",
    "Gel":    "33333333-3333-4333-8333-333333333333",
    "Liquid": "44444444-4444-4444-8444-444444444444",
    "Bar":    "55555555-5555-4555-8555-555555555555",
    "Gummy":  "66666666-6666-4666-8666-666666666666",
    "Capsule":"77777777-7777-4777-8777-777777777777"
}

# ==========================================
# 2. RICH MOCK DATA GENERATION (50 Items)
# ==========================================
print("Generating 50 realistic supplement records...")

categories = [
    {
        "type": "Whey Protein",
        "brands": ["Optimum Nutrition", "Dymatize", "MuscleTech", "MyProtein"],
        "forms": ["Powder"],
        "ingredients_pool": ["Whey Protein Isolate", "Whey Protein Concentrate", "Hydrolyzed Whey", "Cocoa Powder", "Natural Flavors", "Soy Lecithin", "Salt", "Sucralose", "Acesulfame Potassium", "Lactase"],
        "base_macros_100g": {"energy_kcal": 380, "protein_g": 79, "fat_g": 3, "saturated_fat_g": 1.5, "carbohydrate_g": 5, "sugar_g": 2, "added_sugar_g": 0, "sodium_mg": 140, "cholesterol_mg": 40},
        "base_macros_serving": {"energy_kcal": 120, "protein_g": 24, "fat_g": 1, "saturated_fat_g": 0.5, "carbohydrate_g": 3, "sugar_g": 1, "added_sugar_g": 0, "sodium_mg": 50, "cholesterol_mg": 15},
        "desc": "Fast-absorbing whey protein for muscle recovery.",
        "input_type": "webscraper"
    },
    {
        "type": "Vegan Protein",
        "brands": ["Orgain", "Vega", "Sunwarrior", "Garden of Life"],
        "forms": ["Powder"],
        "ingredients_pool": ["Pea Protein", "Brown Rice Protein", "Chia Seed", "Erythritol", "Organic Cocoa", "Sea Salt", "Stevia Leaf Extract", "Guar Gum", "Xanthan Gum"],
        "base_macros_100g": {"energy_kcal": 390, "protein_g": 65, "fat_g": 7, "saturated_fat_g": 1, "carbohydrate_g": 15, "sugar_g": 0, "added_sugar_g": 0, "sodium_mg": 350, "cholesterol_mg": 0},
        "base_macros_serving": {"energy_kcal": 150, "protein_g": 21, "fat_g": 4, "saturated_fat_g": 0, "carbohydrate_g": 8, "sugar_g": 0, "added_sugar_g": 0, "sodium_mg": 180, "cholesterol_mg": 0},
        "desc": "Plant-based protein blend, dairy-free and gluten-free.",
        "input_type": "manual"
    },
    {
        "type": "Pre-Workout",
        "brands": ["Cellucor", "Redcon1", "Legion", "PreKaged"],
        "forms": ["Powder"],
        "ingredients_pool": ["Citrulline Malate", "Beta-Alanine", "Betaine Anhydrous", "Caffeine Anhydrous", "L-Theanine", "Creatine Monohydrate", "Malic Acid", "Silicon Dioxide", "Red Dye #40"],
        "base_macros_100g": {"energy_kcal": 10, "protein_g": 0, "fat_g": 0, "saturated_fat_g": 0, "carbohydrate_g": 2, "sugar_g": 0, "added_sugar_g": 0, "sodium_mg": 50, "caffeine_mg": 2000},
        "base_macros_serving": {"energy_kcal": 5, "protein_g": 0, "fat_g": 0, "saturated_fat_g": 0, "carbohydrate_g": 1, "sugar_g": 0, "added_sugar_g": 0, "sodium_mg": 10, "caffeine_mg": 200},
        "desc": "High stimulant pre-workout for focus and energy.",
        "input_type": "webscraper"
    },
    {
        "type": "Energy Bar",
        "brands": ["Quest", "RXBAR", "Kind", "Clif Bar"],
        "forms": ["Bar"],
        "ingredients_pool": ["Almonds", "Peanuts", "Soluble Corn Fiber", "Milk Protein Isolate", "Unsweetened Chocolate", "Cocoa Butter", "Erythritol", "Sea Salt", "Stevia"],
        "base_macros_100g": {"energy_kcal": 350, "protein_g": 33, "fat_g": 14, "saturated_fat_g": 5, "carbohydrate_g": 35, "sugar_g": 2, "added_sugar_g": 0, "sodium_mg": 350, "cholesterol_mg": 5},
        "base_macros_serving": {"energy_kcal": 180, "protein_g": 21, "fat_g": 7, "saturated_fat_g": 2.5, "carbohydrate_g": 22, "sugar_g": 1, "added_sugar_g": 0, "sodium_mg": 200, "cholesterol_mg": 5},
        "desc": "Low carb protein bar, perfect for on-the-go snacking.",
        "input_type": "manual"
    },
    {
        "type": "Multivitamin",
        "brands": ["Nature Made", "Thorne", "Life Extension", "One A Day"],
        "forms": ["Capsule", "Pill"],
        "ingredients_pool": ["Vitamin A Acetate", "Ascorbic Acid", "Cholecalciferol", "Vitamin E", "Thiamin", "Riboflavin", "Niacinamide", "Vitamin B6", "Folate", "Vitamin B12", "Biotin", "Zinc Oxide"],
        "base_macros_100g": {"energy_kcal": 0, "protein_g": 0, "fat_g": 0, "saturated_fat_g": 0, "carbohydrate_g": 0, "sugar_g": 0, "added_sugar_g": 0, "sodium_mg": 0},
        "base_macros_serving": {"energy_kcal": 0, "protein_g": 0, "fat_g": 0, "saturated_fat_g": 0, "carbohydrate_g": 0, "sugar_g": 0, "added_sugar_g": 0, "sodium_mg": 0, "vitamin_d_iu": 2000, "vitamin_c_mg": 100},
        "desc": "Daily multivitamin for immune support and wellness.",
        "input_type": "webscraper"
    }
]

raw_data = []

def perturb(val):
    """Add slight random variation (±10%) to make data realistic"""
    if isinstance(val, (int, float)) and val > 0:
        return round(val * random.uniform(0.9, 1.1), 1)
    return val

for i in range(50):
    cat = categories[i % len(categories)]
    brand = random.choice(cat["brands"])

    # Randomize Ingredients (Pick 4-8 ingredients)
    num_ing = random.randint(4, 8)
    curr_ingredients = random.sample(cat["ingredients_pool"], k=min(num_ing, len(cat["ingredients_pool"])))

    # Randomize Macros
    macros_100 = {k: perturb(v) for k, v in cat["base_macros_100g"].items()}
    macros_serv = {k: perturb(v) for k, v in cat["base_macros_serving"].items()}

    item = {
        "id": get_uuid(),
        "supplement_dose_form_id": forms.get(cat["forms"][0], forms["Powder"]),
        "supplement_input_type": cat["input_type"],
        "human_in_the_loop": random.choice([True, False]),
        "supplement_name": f"{brand} {cat['type']} {random.choice(['Pro', 'Elite', 'Max', 'Gold', 'Natural'])}",
        "supplement_brand": brand,
        "supplement_description": cat["desc"],
        "supplement_ingredient": json.dumps({"ingredients": curr_ingredients}),
        "nutritional_info_per_100g": json.dumps(macros_100),
        "nutritional_info_per_serving": json.dumps(macros_serv),
        "nutritional_info_per_serving_definition": f"1 Serving ({random.randint(5, 40)}g)",
        "supplement_warning_label": "None",
        "supplement_certifications": random.choice(["NSF", "Informed Choice", "GMP", None]),
        "supplement_additional_information": "Store in a cool, dry place.",
        "supplement_website": f"https://www.{brand.lower().replace(' ','')}.com",
        "batch_testing_org": random.choice(["Labdoor", "NSF", None]),
        "supplement_status": 3,
        "created_on": get_date(random.randint(1, 60)),
        "created_by": "system_seed",
        "last_modified_on": get_date(1),
        "last_modified_by": "admin"
    }
    raw_data.append(item)

df = pd.DataFrame(raw_data)
print(f"Dataframe created: {len(df)} rows.")

# ==========================================
# 3. VECTORIZATION (THE HEAVY LIFTING)
# ==========================================
print("Initializing AI Model for Vectorization...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# --- A. RICH TEXT VECTORIZATION ---
def create_rich_text(row):
    try:
        data = json.loads(row['supplement_ingredient'])
        ing_list = data.get('ingredients', [])
        ing_str = ", ".join(ing_list)
        # Combine: Name + Brand + Description + Ingredients
        return f"{row['supplement_name']} {row['supplement_brand']} {row['supplement_description']} {ing_str}"
    except: return ""

print("Generating Text Vectors...")
df['rich_text_temp'] = df.apply(create_rich_text, axis=1)
text_vectors = model.encode(df['rich_text_temp'].tolist())

# --- B. MACRO VECTORIZATION (Standardize -> Normalize) ---
# We standardize to GRAMS first.
def get_standard_macros(json_str):
    try:
        d = json.loads(json_str)
        # Extract Core Macros (Protein, Carbs, Fat)
        # Note: We can expand this list, but for vector similarity, these 3 are the "Shape" of the food.
        return [
            d.get("protein_g", 0),
            d.get("carbohydrate_g", 0),
            d.get("fat_g", 0)
        ]
    except: return [0, 0, 0]

# Extract from 100g info (Priority 1) or Serving (Priority 2)
def get_best_macros(row):
    try:
        m100 = get_standard_macros(row['nutritional_info_per_100g'])
        if sum(m100) > 0: return m100
    except: pass
    return get_standard_macros(row['nutritional_info_per_serving'])

print("Generating Nutrition Vectors...")
macro_data = np.array(df.apply(get_best_macros, axis=1).tolist())

# Normalize (0 to 1 range)
scaler = MinMaxScaler()
macro_vectors = scaler.fit_transform(macro_data)

# --- C. COMBINE (HYBRID VECTOR) ---
# 50% Text, 50% Nutrition
print("Combining into Hybrid Vectors...")
hybrid_vectors = np.hstack([text_vectors * 0.5, macro_vectors * 0.5])

# Save to DataFrame as stringified lists (for CSV storage)
df['vector_100g_ingredient'] = [str(vec.tolist()) for vec in hybrid_vectors]
# We use the same logic for the per_serving column for this mock
df['vector_perserving_ingredient'] = df['vector_100g_ingredient']

# Cleanup
df.drop(columns=['rich_text_temp'], inplace=True)

# ==========================================
# 4. EXPORT
# ==========================================
filename = "supplements_full_schema_balanced_v3.csv"
df.to_csv(filename, index=False)

print(f"\nSUCCESS! Generated '{filename}'")
print(f"- Rows: {len(df)}")
print(f"- Vector Dimensions: {hybrid_vectors.shape[1]} (384 Text + 3 Macros)")
print(f"- Columns: {list(df.columns)}")
print("You can now load this CSV in your Search Script.")

Generating 50 realistic supplement records...
Dataframe created: 50 rows.
Initializing AI Model for Vectorization...
Generating Text Vectors...
Generating Nutrition Vectors...
Combining into Hybrid Vectors...

SUCCESS! Generated 'supplements_full_schema_balanced_v3.csv'
- Rows: 50
- Vector Dimensions: 387 (384 Text + 3 Macros)
- Columns: ['id', 'supplement_dose_form_id', 'supplement_input_type', 'human_in_the_loop', 'supplement_name', 'supplement_brand', 'supplement_description', 'supplement_ingredient', 'nutritional_info_per_100g', 'nutritional_info_per_serving', 'nutritional_info_per_serving_definition', 'supplement_warning_label', 'supplement_certifications', 'supplement_additional_information', 'supplement_website', 'batch_testing_org', 'supplement_status', 'created_on', 'created_by', 'last_modified_on', 'last_modified_by', 'vector_100g_ingredient', 'vector_perserving_ingredient']
You can now load this CSV in your Search Script.


# Vector and Similarity Search

## Test 8

In [ ]:
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# 1. SETUP & CONFIGURATION
# ==========================================
FILENAME = "supplements_full_schema_balanced_v3.csv"

# CONSTANT: The Fixed "Universe" for Vector Math (30 Common Dimensions)
# Vectors MUST be fixed length. Rare/Unique keys will fallback to Text Vector.
FIXED_VECTOR_SCHEMA = [
    "energy_kcal", "protein_g", "carbohydrate_g", "sugar_g", "added_sugar_g",
    "fat_g", "saturated_fat_g", "trans_fat_g", "cholesterol_mg", "sodium_mg",
    "fiber_g", "caffeine_mg",
    # Vitamins & Minerals
    "vitamin_a_mcg", "vitamin_c_mg", "vitamin_d_mcg", "vitamin_e_mg", "vitamin_k_mcg",
    "thiamin_mg", "riboflavin_mg", "niacin_mg", "vitamin_b6_mg", "folate_mcg", "vitamin_b12_mcg",
    "biotin_mcg", "pantothenic_acid_mg",
    "calcium_mg", "iron_mg", "magnesium_mg", "zinc_mg", "potassium_mg"
]

try:
    df = pd.read_csv(FILENAME)
    print(f"Loaded Database: {len(df)} products.")
except FileNotFoundError:
    print(f"Error: {FILENAME} not found. Please run the generation script first.")
    exit()

print("Loading AI Model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

Loaded Database: 50 products.
Loading AI Model...


In [ ]:
# 2. SMART MAPPING ENGINE
# ==========================================

def map_key_to_schema(raw_key):
    """
    smart fuzzy mapper. Converts weird OCR keys to our Fixed Schema.
    Examples:
      "Vit D3 (IU)" -> "vitamin_d_mcg"
      "B-12" -> "vitamin_b12_mcg"
      "Sodium" -> "sodium_mg"
    """
    k = raw_key.lower().replace("-", "").replace(" ", "")

    # Dictionary of distinct markers
    if "energy" in k or "calor" in k: return "energy_kcal"
    if "added" in k and "sugar" in k: return "added_sugar_g"
    if "sugar" in k: return "sugar_g"
    if "fiber" in k: return "fiber_g"
    if "saturated" in k: return "saturated_fat_g"
    if "trans" in k: return "trans_fat_g"
    if "fat" in k: return "fat_g"
    if "cholesterol" in k: return "cholesterol_mg"
    if "sodium" in k: return "sodium_mg"
    if "potassium" in k: return "potassium_mg"
    if "protein" in k: return "protein_g"
    if "carb" in k: return "carbohydrate_g"
    if "caffeine" in k: return "caffeine_mg"

    # Vitamins Fuzzy Matching
    if "vit" in k or "cholecalciferol" in k or "retinol" in k or "ascorbic" in k:
        if "d" in k: return "vitamin_d_mcg" # Catches D, D3, Vitamin D
        if "a" in k and "panto" not in k: return "vitamin_a_mcg"
        if "c" in k and "calcium" not in k: return "vitamin_c_mg"
        if "e" in k: return "vitamin_e_mg"
        if "k" in k: return "vitamin_k_mcg"
        if "b12" in k or "cobalamin" in k: return "vitamin_b12_mcg"
        if "b6" in k or "pyridox" in k: return "vitamin_b6_mg"

    if "thiamin" in k or "b1" in k: return "thiamin_mg"
    if "riboflavin" in k or "b2" in k: return "riboflavin_mg"
    if "niacin" in k or "b3" in k: return "niacin_mg"
    if "folate" in k or "folic" in k: return "folate_mcg"
    if "biotin" in k: return "biotin_mcg"

    # Minerals
    if "calcium" in k: return "calcium_mg"
    if "iron" in k: return "iron_mg"
    if "magnesium" in k: return "magnesium_mg"
    if "zinc" in k: return "zinc_mg"

    return None # Key not in fixed schema (will be handled by Text Vector)

def process_nutrition(input_data):
    """
    Returns:
      1. standardized_vector (list): Values for the fixed math schema
      2. rare_text_dump (str): Text string of any nutrients that didn't fit the schema
    """
    vector_map = {key: 0.0 for key in FIXED_VECTOR_SCHEMA}
    rare_nutrients = []

    if not isinstance(input_data, dict):
        return list(vector_map.values()), ""

    for raw_key, value in input_data.items():
        try: val = float(value)
        except: continue

        clean_key = raw_key.lower().strip()
        target_field = map_key_to_schema(clean_key)

        # --- UNIT NORMALIZATION ---
        multiplier = 1.0

        # IU Logic
        if "iu" in clean_key or "i.u." in clean_key:
            if target_field == "vitamin_d_mcg": multiplier = 0.025
            elif target_field == "vitamin_a_mcg": multiplier = 0.3
            elif target_field == "vitamin_e_mg": multiplier = 0.67
            else: multiplier = 0 # Ignore unknown IU

        # Standard Mass Logic
        elif "mcg" in clean_key:
            # If target expects mg, divide by 1000
            if target_field and "_mg" in target_field: multiplier = 0.001
            # If target expects mcg, keep 1.0
        elif "mg" in clean_key:
            # If target expects mcg, multiply by 1000
            if target_field and "_mcg" in target_field: multiplier = 1000.0
        elif "g" in clean_key and "mg" not in clean_key:
            # If target expects mg, multiply by 1000
            if target_field and "_mg" in target_field: multiplier = 1000.0

        # --- ASSIGNMENT ---
        if target_field:
            vector_map[target_field] = val * multiplier
        else:
            # Rare/Unique Nutrient? Add to Text Description!
            rare_nutrients.append(f"{raw_key}: {val}")

    vector_values = [vector_map[k] for k in FIXED_VECTOR_SCHEMA]
    return vector_values, ", ".join(rare_nutrients)

In [ ]:
# 3. RE-VECTORIZATION
# ==========================================
print("Standardizing Database...")

nutrition_vectors_list = []
text_content_list = []

for idx, row in df.iterrows():
    # 1. Parse JSON
    try: n100 = json.loads(row['nutritional_info_per_100g'])
    except: n100 = {}
    try: nserv = json.loads(row['nutritional_info_per_serving'])
    except: nserv = {}

    # Use 100g if valid, else serving
    source_nut = n100 if any(v > 0 for v in n100.values()) else nserv

    # 2. Process Nutrition
    vec, rare_text = process_nutrition(source_nut)
    nutrition_vectors_list.append(vec)

    # 3. Process Text (Include Ingredients + Rare Nutrients)
    try:
        ing_data = json.loads(row['supplement_ingredient'])
        ing_str = ", ".join(ing_data.get("ingredients", [])) if isinstance(ing_data.get("ingredients"), list) else ""
    except: ing_str = ""

    # Rich Text = Name + Brand + Desc + Ingredients + Rare Nutrients
    full_text = f"{ing_str} {rare_text}"
    text_content_list.append(full_text)

# Create Vectors
scaler = MinMaxScaler()
nut_vecs = scaler.fit_transform(np.array(nutrition_vectors_list))
txt_vecs = model.encode(text_content_list)
hybrid_vectors = np.hstack([txt_vecs * 0.5, nut_vecs * 0.5])

Standardizing Database...


In [ ]:
# 4. INTERACTIVE SEARCH
# ==========================================
def find_alternatives(json_payload, threshold=60.0, self_match_threshold=99.0):
    try:
        data = json.loads(json_payload)

        # ... (Vectorization Logic remains the same) ...
        q_ing = data.get("ingredients", "")
        if isinstance(q_ing, list): q_ing = ", ".join(q_ing)
        q_nut = data.get("nutrition", {})

        # Process Query Vectors (Same as before)
        q_vec_raw, q_rare_text = process_nutrition(q_nut)
        q_nut_vec = scaler.transform(np.array([q_vec_raw]))
        q_text_final = f"{q_ing} {q_rare_text}"
        q_txt_vec = model.encode([q_text_final])
        q_hybrid = np.hstack([q_txt_vec * 0.5, q_nut_vec * 0.5])

        # Calculate Scores
        scores = cosine_similarity(q_hybrid, hybrid_vectors)[0] * 100

        results = df.copy()
        results['score'] = scores

        # --- NEW LOGIC: SEPARATE "SELF" FROM "ALTERNATIVES" ---

        # 1. Identify "Self Matches" (Score > 90%)
        # These are likely the product itself or identical re-brands
        self_matches = results[results['score'] >= self_match_threshold]

        # 2. Identify "Alternatives" (Score between Threshold and Self-Threshold)
        # e.g., 60% < Score < 90%
        alternatives = results[
            (results['score'] >= threshold) &
            (results['score'] < self_match_threshold)
        ]

        # Sort
        alternatives = alternatives.sort_values('score', ascending=False).head(5)

        # --- DISPLAY ---

        # Optional: Tell the user what we identified the product as
        if not self_matches.empty:
            top_self = self_matches.sort_values('score', ascending=False).iloc[0]
            print(f"\n[INFO] Identified Scanned Product as:")
            print(f"       '{top_self['supplement_name']}' ({top_self['score']:.1f}% Match)")
            print(f"       (Excluded from alternatives list)")

        if alternatives.empty:
            print(f"\nNo alternatives found (between {threshold}% - {self_match_threshold}%).")
        else:
            print(f"\nFound {len(alternatives)} alternatives:\n")
            for i, row in alternatives.iterrows():
                print(f"> {row['score']:.1f}% | {row['supplement_name']} ({row['supplement_brand']})")
                print(f"  Desc: {row['supplement_description'][:60]}...")
                print("-" * 30)

    except json.JSONDecodeError: print("Invalid JSON.")

In [ ]:
# 5. EXECUTION
# ==========================================
if __name__ == "__main__":
    print("\nPaste JSON Query below:")
    # Using Loop to keep reading lines until bracket closes, or simple input
    raw_input = input()
    find_alternatives(raw_input)


Paste JSON Query below:
{"ingredients": ["Thiamin", "Cholecalciferol", "Folate", "Ascorbic Acid"], "nutrition": {"energy_kcal": 0, "protein_g": 0, "fat_g": 0, "saturated_fat_g": 0, "carbohydrate_g": 0, "sugar_g": 0, "added_sugar_g": 0, "sodium_mg": 0, "vitamin_d_iu": 2033.2, "vitamin_c_mg": 106.3}}

[INFO] Identified Scanned Product as:
       'One A Day Multivitamin Max' (100.0% Match)
       (Excluded from alternatives list)

Found 5 alternatives:

> 89.1% | Life Extension Multivitamin Natural (Life Extension)
  Desc: Daily multivitamin for immune support and wellness....
------------------------------
> 89.0% | One A Day Multivitamin Gold (One A Day)
  Desc: Daily multivitamin for immune support and wellness....
------------------------------
> 88.9% | One A Day Multivitamin Natural (One A Day)
  Desc: Daily multivitamin for immune support and wellness....
------------------------------
> 86.7% | Nature Made Multivitamin Max (Nature Made)
  Desc: Daily multivitamin for immune suppo

## Test 9 (Hugging face)

In [ ]:
import pandas as pd
import numpy as np
import json
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# 1. SETUP & CONFIGURATION
# ==========================================
FILENAME = "supplements_full_schema_balanced_v3.csv"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Fixed Nutrition Schema (Vector Dimensions)
FIXED_VECTOR_SCHEMA = [
    "energy_kcal", "protein_g", "carbohydrate_g", "sugar_g", "added_sugar_g",
    "fat_g", "saturated_fat_g", "trans_fat_g", "cholesterol_mg", "sodium_mg",
    "fiber_g", "caffeine_mg",
    "vitamin_a_mcg", "vitamin_c_mg", "vitamin_d_mcg", "vitamin_e_mg", "vitamin_k_mcg",
    "thiamin_mg", "riboflavin_mg", "niacin_mg", "vitamin_b6_mg", "folate_mcg", "vitamin_b12_mcg",
    "biotin_mcg", "pantothenic_acid_mg", "calcium_mg", "iron_mg", "magnesium_mg", "zinc_mg", "potassium_mg"
]

In [ ]:
# 2. HUGGING FACE EMBEDDER (The Replacement)
# ==========================================
class HFEmbedder:
    def __init__(self, model_name):
        print(f"Loading HF Model: {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

    def encode(self, texts):
        """Generates embeddings matching SentenceTransformer's output"""
        # 1. Tokenize
        inputs = self.tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

        # 2. Model Inference (No Gradient needed for inference)
        with torch.no_grad():
            outputs = self.model(**inputs)

        # 3. Mean Pooling (Average of all token vectors)
        # attention_mask ensures we don't average padding tokens
        embeddings = self._mean_pooling(outputs, inputs['attention_mask'])

        # 4. Normalize (Cosine Similarity requires normalized vectors)
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        return embeddings.numpy()

    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Initialize AI
embedder = HFEmbedder(MODEL_NAME)

# Load Data
try:
    df = pd.read_csv(FILENAME)
    print(f"Loaded Database: {len(df)} products.")
except FileNotFoundError:
    print("Error: CSV not found."); exit()

Loading HF Model: sentence-transformers/all-MiniLM-L6-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded Database: 50 products.


In [ ]:
# 3. LOGIC: MAPPING & NORMALIZATION
# ==========================================
def map_key_to_schema(raw_key):
    k = raw_key.lower().replace("-", "").replace(" ", "")

    # Macros
    if "energy" in k or "calor" in k: return "energy_kcal"
    if "added" in k and "sugar" in k: return "added_sugar_g"
    if "sugar" in k: return "sugar_g"
    if "fiber" in k: return "fiber_g"
    if "saturated" in k: return "saturated_fat_g"
    if "trans" in k: return "trans_fat_g"
    if "fat" in k: return "fat_g"
    if "protein" in k: return "protein_g"
    if "carb" in k: return "carbohydrate_g"

    # Micros
    if "cholesterol" in k: return "cholesterol_mg"
    if "sodium" in k: return "sodium_mg"
    if "potassium" in k: return "potassium_mg"
    if "caffeine" in k: return "caffeine_mg"

    # Vitamins
    if "vit" in k or "cholecalciferol" in k or "retinol" in k or "ascorbic" in k:
        if "d" in k: return "vitamin_d_mcg"
        if "a" in k and "panto" not in k: return "vitamin_a_mcg"
        if "c" in k and "calcium" not in k: return "vitamin_c_mg"
        if "e" in k: return "vitamin_e_mg"
        if "k" in k: return "vitamin_k_mcg"
        if "b12" in k: return "vitamin_b12_mcg"
        if "b6" in k: return "vitamin_b6_mg"

    if "thiamin" in k: return "thiamin_mg"
    if "riboflavin" in k: return "riboflavin_mg"
    if "niacin" in k: return "niacin_mg"
    if "folate" in k: return "folate_mcg"
    if "biotin" in k: return "biotin_mcg"

    # Minerals
    if "calcium" in k: return "calcium_mg"
    if "iron" in k: return "iron_mg"
    if "magnesium" in k: return "magnesium_mg"
    if "zinc" in k: return "zinc_mg"

    return None

def process_nutrition(input_data):
    vector_map = {key: 0.0 for key in FIXED_VECTOR_SCHEMA}
    rare_nutrients = []

    if not isinstance(input_data, dict): return list(vector_map.values()), ""

    for raw_key, value in input_data.items():
        try: val = float(value)
        except: continue

        clean_key = raw_key.lower().strip()
        target = map_key_to_schema(clean_key)

        # Unit Standardization
        mult = 1.0
        if "iu" in clean_key:
            if target == "vitamin_d_mcg": mult = 0.025
            elif target == "vitamin_a_mcg": mult = 0.3
            elif target == "vitamin_e_mg": mult = 0.67
            else: mult = 0
        elif "mcg" in clean_key:
            if target and "_mg" in target: mult = 0.001
        elif "mg" in clean_key:
            if target and "_mcg" in target: mult = 1000.0
        elif "g" in clean_key and "mg" not in clean_key:
            if target and "_mg" in target: mult = 1000.0

        if target:
            vector_map[target] = val * mult
        else:
            rare_nutrients.append(f"{raw_key}: {val}")

    return [vector_map[k] for k in FIXED_VECTOR_SCHEMA], ", ".join(rare_nutrients)

In [ ]:
# 4. VECTORIZATION (DB PREP)
# ==========================================
print("Standardizing Database...")

nut_vecs_list = []
text_vecs_list = []

for _, row in df.iterrows():
    # 1. Get Nutrition (100g Priority)
    try: n100 = json.loads(row['nutritional_info_per_100g'])
    except: n100 = {}
    try: nserv = json.loads(row['nutritional_info_per_serving'])
    except: nserv = {}

    src = n100 if any(v > 0 for v in n100.values()) else nserv
    vec, rare_txt = process_nutrition(src)
    nut_vecs_list.append(vec)

    # 2. Get Text (Ingredients + Rare Nutrients)
    try:
        ing = json.loads(row['supplement_ingredient'])
        ing_s = ", ".join(ing.get("ingredients", [])) if isinstance(ing.get("ingredients"), list) else ""
    except: ing_s = ""

    text_vecs_list.append(f"{ing_s} {rare_txt}")

# Compute Vectors
scaler = MinMaxScaler()
db_nut_vecs = scaler.fit_transform(np.array(nut_vecs_list))
db_text_vecs = embedder.encode(text_vecs_list)

# Hybrid Combine (50/50)
db_hybrid_vecs = np.hstack([db_text_vecs * 0.5, db_nut_vecs * 0.5])

Standardizing Database...


In [ ]:
# 5. SEARCH FUNCTION
# ==========================================
def find_alternatives(json_payload, threshold=60.0, self_match_threshold=99.0):
    try:
        data = json.loads(json_payload)

        # Parse Inputs
        q_ing = data.get("ingredients", "")
        if isinstance(q_ing, list): q_ing = ", ".join(q_ing)
        q_nut = data.get("nutrition", {})

        # Process Query
        q_nut_raw, q_rare_text = process_nutrition(q_nut)
        q_nut_vec = scaler.transform(np.array([q_nut_raw]))

        q_text = f"{q_ing} {q_rare_text}"
        q_txt_vec = embedder.encode([q_text])

        q_hybrid = np.hstack([q_txt_vec * 0.5, q_nut_vec * 0.5])

        # Search
        scores = cosine_similarity(q_hybrid, db_hybrid_vecs)[0] * 100

        # Filter Results
        results = df.copy()
        results['score'] = scores

        # Separate Self vs Alternatives
        self_match = results[results['score'] >= self_match_threshold]
        alternatives = results[(results['score'] >= threshold) & (results['score'] < self_match_threshold)]

        # Display
        if not self_match.empty:
            top = self_match.iloc[0]
            print(f"\n[INFO] Identified Product: '{top['supplement_name']}' ({top['score']:.2f}%)")

        top_alts = alternatives.sort_values('score', ascending=False).head(5)

        if top_alts.empty:
            print(f"\nNo alternatives found (> {threshold}%).")
        else:
            print(f"\nFound {len(top_alts)} Alternatives:")
            for _, row in top_alts.iterrows():
                print(f"> {row['score']:.1f}% | {row['supplement_name']} ({row['supplement_brand']})")
                print(f"  Desc: {row['supplement_description'][:60]}...")
                print("-" * 30)

    except json.JSONDecodeError: print("Invalid JSON.")
    except Exception as e: print(f"Error: {e}")

In [ ]:
# 6. RUN
# ==========================================
if __name__ == "__main__":
    print("\nPaste JSON Query below:")
    raw_input = input()
    find_alternatives(raw_input)


Paste JSON Query below:
{"ingredients": ["Red Dye #40", "Malic Acid", "Caffeine Anhydrous", "Beta-Alanine", "Silicon Dioxide", "Betaine Anhydrous"], "nutrition": {"energy_kcal": 10.1, "protein_g": 0, "fat_g": 0, "saturated_fat_g": 0, "carbohydrate_g": 1.9, "sugar_g": 0, "added_sugar_g": 0, "sodium_mg": 45.5, "caffeine_mg": 1893.5}}

[INFO] Identified Product: 'Legion Pre-Workout Gold' (100.00%)

Found 5 Alternatives:
> 94.9% | Cellucor Pre-Workout Pro (Cellucor)
  Desc: High stimulant pre-workout for focus and energy....
------------------------------
> 94.8% | Cellucor Pre-Workout Max (Cellucor)
  Desc: High stimulant pre-workout for focus and energy....
------------------------------
> 94.7% | Legion Pre-Workout Max (Legion)
  Desc: High stimulant pre-workout for focus and energy....
------------------------------
> 94.1% | Legion Pre-Workout Pro (Legion)
  Desc: High stimulant pre-workout for focus and energy....
------------------------------
> 94.0% | Redcon1 Pre-Workout Max (Red